# Check synced embeddings

After `sync_embeddings` (`src/python/extract_embeddings/sync_embeddings.py`) brought the extracted
embeddings in line with the regenerated `patch_coordinates.h5`, this notebook checks every slide of
the extraction config(s) in `CONFIGS`:

1. **Cell ids.** `embeddings_dataset.h5` must hold exactly the cells of the current
   `patch_coordinates.h5`, **row for row**, with the same labels. Every row must be written, and
   every dataset in the file must have the same length. The table also flags what a sync can leave
   behind: a `_sync_tmp/` folder or `*.sync_tmp` file from a killed run, and `--keep-old` backups.
2. **Maps.** PCA and UMAP of the **central token**, per slide. The central token is the mean of the 4
   central tokens (`top_left`, `top_right`, `bottom_left`, `bottom_right`), which the classifier reads.
   Cells are coloured by whether they were **kept** from the original extraction or **added** by the
   sync, and by cell type. Kept vs added comes from the `cell_id_drift.csv` that
   `scripts/check_cell_id_drift.py --ids-csv` wrote *before* the sync. Re-running that script with the
   same file name overwrites it with an empty list.

Added cells are extracted with the same model and settings, so they should fall inside the kept cells'
cloud. An island made only of added cells would point to an extraction mismatch. Some shift is
legitimate, though: the cells the earlier processing left out can differ biologically.

Reads only ids, labels and a sample of rows, with no torch, so it runs on a login node.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))


def _load_dotenv(path: Path) -> None:
    """Minimal `.env` loader (KEY=value per line), as in batch_effects_across_models.ipynb."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip()
        if value:
            os.environ.setdefault(key, value)


_load_dotenv(REPO_ROOT / ".env")

In [ ]:
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# torch-free: resolves slides like the extraction does, without importing extract_embeddings.py
from src.python.extract_embeddings.sync_embeddings import BACKUP_NAME, MERGED_SUFFIX, TMP_DIR, _hash, _light_runs

try:
    import umap
except ImportError:
    umap = None
    print("umap-learn is not installed (`pip install umap-learn`): only the PCA panels are drawn.")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

## Settings

In [ ]:
# extraction config(s) that were synced; relative paths are resolved from the repo root
CONFIGS = ["configs/extract_embeddings/uni2_224.json"]
# check_cell_id_drift.py --ids-csv output written BEFORE the sync: tells which cells the sync added
DRIFT_CSV = REPO_ROOT / "cell_id_drift.csv"

CENTRAL_KEYS = ["top_left", "top_right", "bottom_left", "bottom_right"]   # averaged into the central token

PLOT_SLIDES = "changed"   # "changed": slides the sync added cells to (from DRIFT_CSV) | "all" | a list of names
N_KEPT, N_ADDED = 5000, 5000   # cells sampled per slide from each group (all of them when fewer)
N_PCS = 50                     # PCA components; UMAP runs on these
UMAP_KWARGS = dict(n_neighbors=15, min_dist=0.3, random_state=0)
KNN_K = 15                     # neighbours for the added-neighbour enrichment
SEED = 0

In [ ]:
runs = {}
for config in CONFIGS:
    _, config_runs = _light_runs(REPO_ROOT / config)
    for run in config_runs:
        runs.setdefault(run["output_path"], run)
slides = pd.DataFrame([
    {"sample": Path(out).name, "extraction": Path(out).parent.parent.name,
     "emb_path": Path(out) / "embeddings_dataset.h5",
     "coords_path": Path(run["dataset_configs"]["cells_info_path"])}
    for out, run in runs.items()
])
print(f"{len(slides)} slides in {', '.join(map(str, CONFIGS))} -> {', '.join(slides['extraction'].unique())}")

## 1. Cell ids: `embeddings_dataset.h5` vs `patch_coordinates.h5`

| status | meaning |
|---|---|
| `ok` | same cell ids in the same order, same labels, every row written, all datasets the same length |
| `LABELS_DIFFER` | same rows, but some `cell_labels` differ from the current `cell_type` |
| `ORDER_DIFFERS` | same cells, different row order |
| `IDS_DIFFER` | cells missing from one file or the other (`only_emb` / `only_coords`) or duplicated |
| `UNFINISHED` | empty `cell_ids` rows: an extraction that never finished |
| `BAD_LENGTHS` | a dataset in the file doesn't have one row per cell |
| `NOT_EXTRACTED` / `NO_COORDS` / `UNREADABLE` | a file is missing or can't be opened |

`leftovers` lists what a sync leaves behind: a `_sync_tmp/` folder or `*.sync_tmp` file from a killed
run (re-running `sync_embeddings` cleans those up), and `backup` when `--keep-old` kept the previous file.

In [ ]:
def check_slide(emb_path: Path, coords_path: Path) -> dict:
    out = emb_path.parent
    leftovers = [name for name in (TMP_DIR, "embeddings_dataset.h5" + MERGED_SUFFIX) if (out / name).exists()]
    row = {"leftovers": ", ".join(leftovers), "backup": (out / BACKUP_NAME).exists()}
    if not coords_path.exists():
        return {**row, "status": "NO_COORDS"}
    with h5py.File(coords_path, "r", locking=False) as f:
        c_ids, c_labels = f["cell_id"][()], f["cell_type"][()]
    row["n_coords"] = len(c_ids)
    if not emb_path.exists():
        return {**row, "status": "NOT_EXTRACTED"}
    try:
        with h5py.File(emb_path, "r", locking=False) as f:
            e_ids, e_labels = f["cell_ids"][()], f["cell_labels"][()]
            lengths = {k: d.shape[0] for k, d in f.items() if isinstance(d, h5py.Dataset)}
    except (OSError, KeyError) as e:
        return {**row, "status": "UNREADABLE", "detail": f"{type(e).__name__}: {e}"}

    eh, ch = _hash(e_ids), _hash(c_ids)
    row.update(
        n_emb=len(e_ids),
        n_unwritten=int(sum(len(v) == 0 for v in e_ids)),
        only_emb=int((~np.isin(eh, ch)).sum()),
        only_coords=int((~np.isin(ch, eh)).sum()),
        dup_emb=len(eh) - len(np.unique(eh)),
        dup_coords=len(ch) - len(np.unique(ch)),
    )
    same_rows = len(eh) == len(ch) and bool(np.array_equal(eh, ch))
    row["label_diff"] = int((_hash(e_labels) != _hash(c_labels)).sum()) if same_rows else None
    bad_lengths = sorted(k for k, n in lengths.items() if n != len(e_ids))
    if bad_lengths:
        return {**row, "status": "BAD_LENGTHS", "detail": ", ".join(f"{k}: {lengths[k]:,}" for k in bad_lengths)}
    if row["n_unwritten"]:
        return {**row, "status": "UNFINISHED"}
    if same_rows:
        return {**row, "status": "ok" if row["label_diff"] == 0 else "LABELS_DIFFER"}
    if not (row["only_emb"] or row["only_coords"] or row["dup_emb"] or row["dup_coords"]):
        return {**row, "status": "ORDER_DIFFERS"}
    return {**row, "status": "IDS_DIFFER"}


t0, rows = time.perf_counter(), []
for i, s in enumerate(slides.itertuples(), 1):
    rows.append({"sample": s.sample, "extraction": s.extraction, **check_slide(s.emb_path, s.coords_path)})
    print(f"\r{i}/{len(slides)} {s.sample:<70}", end="", flush=True)
print(f"\rchecked {len(slides)} slides in {time.perf_counter() - t0:.0f} s" + " " * 70)

columns = ["sample", "extraction", "status", "n_emb", "n_coords", "only_emb", "only_coords", "dup_emb", "dup_coords",
           "n_unwritten", "label_diff", "leftovers", "backup", "detail"]
checks = pd.DataFrame(rows).reindex(columns=columns)
for c in ["n_emb", "n_coords", "only_emb", "only_coords", "dup_emb", "dup_coords", "n_unwritten", "label_diff"]:
    checks[c] = checks[c].astype("Int64")
checks["leftovers"] = checks["leftovers"].fillna("")
checks["detail"] = checks["detail"].fillna("")

display(checks["status"].value_counts().rename("slides").to_frame())
problems = checks[(checks["status"] != "ok") | (checks["leftovers"] != "")]
if problems.empty:
    print(f"All {len(checks)} slides: ids and labels match patch_coordinates.h5 row for row, no leftovers.")
else:
    print(f"{len(problems)} slide(s) need a look:")
    display(problems)
if checks["backup"].any():
    print(f"{int(checks['backup'].sum())} slide(s) still have a --keep-old backup ({BACKUP_NAME}); "
          f"delete them once you're happy with the maps below.")

## 2. Central-token maps, per slide

For each slide in `PLOT_SLIDES`, up to `N_KEPT` kept and `N_ADDED` added cells are sampled. The
groups are balanced so the added cells stay visible. Their central token goes through PCA
(`N_PCS` components), then UMAP on those PCs. The panels are PCA and UMAP coloured by **kept / added**,
then PCA and UMAP coloured by **cell type**. Kept and added cells are drawn in random order: where they
mix, the cloud looks speckled; an island of added cells shows as solid blue. Cell type uses the slide's 3
most common types, with the rest grey: past three colours, scatter points stop being reliably told
apart, colour-blind readers included. Slides without added cells only get the cell-type panels.

**Added-neighbour enrichment** sums up each slide. Take each added cell's `KNN_K` nearest neighbours
in PCA space and compute the share of them that are also added cells. Divide that by the share
expected if added cells were spread like kept ones. **~1** means the added cells mix with the kept
ones; the higher the number, the more they keep to themselves. A moderate excess can be biology,
since the newly included cells may be a different population. An island of added cells in the maps
is what would point to an extraction problem.

In [ ]:
added_ids: dict[str, set] = {}
if DRIFT_CSV.exists():
    drift = pd.read_csv(DRIFT_CSV, usecols=["sample", "cell_id", "change"], dtype=str)
    drift = drift[drift["change"] == "added"]
    added_ids = {sample: set(g["cell_id"]) for sample, g in drift.groupby("sample")}
    del drift
    print(f"{DRIFT_CSV.name}: {sum(map(len, added_ids.values())):,} added cells on {len(added_ids)} slides")
else:
    print(f"{DRIFT_CSV} not found: cells can't be split into kept / added, the maps show cell types only.")

readable = checks[~checks["status"].isin(["NO_COORDS", "NOT_EXTRACTED", "UNREADABLE", "BAD_LENGTHS"])]
if PLOT_SLIDES == "changed":
    to_plot = readable[readable["sample"].isin(added_ids)] if added_ids else readable
elif PLOT_SLIDES == "all":
    to_plot = readable
else:
    to_plot = readable[readable["sample"].isin(PLOT_SLIDES)]
print(f"{len(to_plot)} slide(s) to map")

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GREY = "#c9c8c1"      # kept cells; cell types past the top N_TYPES
N_TYPES = 3           # categorical hues per scatter panel; the rest folds into "other"


def _decode(values) -> np.ndarray:
    return np.array([v.decode("utf-8") if isinstance(v, bytes) else str(v) for v in values], dtype=object)


def sample_slide(emb_path: Path, added: set) -> dict:
    """Central token (mean of CENTRAL_KEYS) of up to N_KEPT kept + N_ADDED added random cells."""
    rng = np.random.default_rng(SEED)
    with h5py.File(emb_path, "r", locking=False) as f:
        ids, labels = _decode(f["cell_ids"][()]), _decode(f["cell_labels"][()])
        is_added = np.fromiter((i in added for i in ids), dtype=bool, count=len(ids))
        pools = [np.flatnonzero(~is_added), np.flatnonzero(is_added)]
        rows = np.sort(np.concatenate([rng.choice(p, min(n, len(p)), replace=False)
                                       for p, n in zip(pools, (N_KEPT, N_ADDED))]))
        x = np.zeros((len(rows), f[f"embeddings_{CENTRAL_KEYS[0]}"].shape[1]), dtype=np.float64)
        zero_rows = np.zeros(len(rows), dtype=bool)
        for key in CENTRAL_KEYS:
            ds = f[f"embeddings_{key}"]
            # row by row: random rows of a contiguous dataset read faster this way than by fancy indexing
            block = np.stack([ds[i] for i in rows]).astype(np.float64)
            zero_rows |= ~block.any(axis=1)
            x += block
    x /= len(CENTRAL_KEYS)
    return {"x": x.astype(np.float32), "added": is_added[rows], "labels": labels[rows], "n_cells": len(ids),
            "n_added": int(is_added.sum()), "zero_rows": int(zero_rows.sum()),
            "nan_rows": int((~np.isfinite(x)).any(axis=1).sum())}


def embed(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray | None]:
    pca = PCA(n_components=min(N_PCS, x.shape[1], len(x) - 1), random_state=SEED).fit(x)
    pcs = pca.transform(x)
    um = None
    if umap is not None and len(x) > UMAP_KWARGS["n_neighbors"] + 1:
        with warnings.catch_warnings():   # "n_jobs value 1 overridden ... by setting random_state": expected
            warnings.filterwarnings("ignore", message="n_jobs value")
            um = umap.UMAP(**UMAP_KWARGS).fit_transform(pcs)
    return pcs, pca.explained_variance_ratio_, um


def added_neighbour_enrichment(pcs: np.ndarray, added: np.ndarray) -> float:
    if added.sum() < 2 or (~added).sum() < 1:
        return np.nan
    neighbours = NearestNeighbors(n_neighbors=KNN_K + 1).fit(pcs).kneighbors(pcs[added], return_distance=False)[:, 1:]
    expected = (added.sum() - 1) / (len(added) - 1)
    return float(added[neighbours].mean() / expected)


def point_size(n: int) -> float:
    return float(np.clip(4e4 / max(n, 1), 0.3, 8))


def _finish(ax, title: str, handles: list) -> None:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=8.5)
    ax.legend(handles=handles, fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))


def _dot(color: str, label: str) -> Line2D:
    return Line2D([], [], ls="", marker="o", ms=5, color=color, label=label)


def scatter_origin(ax, xy: np.ndarray, added: np.ndarray, title: str) -> None:
    """Kept cells grey, added cells in the first palette colour, drawn in random order: where they
    mix the cloud looks speckled, an island of added cells shows as solid colour."""
    idx = np.random.default_rng(SEED).permutation(len(xy))
    ax.scatter(xy[idx, 0], xy[idx, 1], c=np.where(added[idx], PALETTE[0], GREY), s=point_size(len(xy)),
               linewidths=0, rasterized=True)
    _finish(ax, title, [_dot(PALETTE[0], f"added ({added.sum():,})"), _dot(GREY, f"kept ({(~added).sum():,})")])


def scatter_types(ax, xy: np.ndarray, labels: np.ndarray, title: str) -> None:
    """The N_TYPES most common cell types in fixed palette order, the rest grey underneath."""
    counts = pd.Series(labels).value_counts()
    top = list(counts.index[:N_TYPES])
    colors = dict(zip(top, PALETTE))
    other = ~np.isin(labels, top)
    s = point_size(len(xy))
    ax.scatter(xy[other, 0], xy[other, 1], c=GREY, s=s, linewidths=0, rasterized=True)
    idx = np.random.default_rng(SEED).permutation(np.flatnonzero(~other))   # no type systematically on top
    ax.scatter(xy[idx, 0], xy[idx, 1], c=[colors[lab] for lab in labels[idx]], s=s, linewidths=0, rasterized=True)
    handles = [_dot(colors[lab], f"{lab} ({counts[lab]:,})") for lab in top]
    if other.any():
        handles.append(_dot(GREY, f"{len(counts) - len(top)} other types ({other.sum():,})"))
    _finish(ax, title, handles)

In [ ]:
summary = []
for s in to_plot.itertuples():
    t0 = time.perf_counter()
    d = sample_slide(Path(slides.loc[s.Index, "emb_path"]), added_ids.get(s.sample, set()))
    pcs, var, um = embed(d["x"])
    added = d["added"]
    enrichment = added_neighbour_enrichment(pcs, added)
    summary.append({
        "sample": s.sample, "cells": d["n_cells"], "added by sync": d["n_added"],
        "sampled kept": int((~added).sum()), "sampled added": int(added.sum()),
        "PC1": var[0], "PC2": var[1], "added-neighbour enrichment": enrichment,
        "sampled rows with an all-zero token": d["zero_rows"], "sampled rows with NaN": d["nan_rows"],
        "seconds": time.perf_counter() - t0,
    })

    pc_title = f"PCA (PC1 {var[0]:.1%}, PC2 {var[1]:.1%})"
    panels = [("PCA", pcs[:, :2], pc_title)] + ([("UMAP", um, "UMAP")] if um is not None else [])
    has_added = bool(added.any())
    n_panels = len(panels) * (2 if has_added else 1)
    fig, axes = plt.subplots(1, n_panels, figsize=(4.6 * n_panels, 3.9), squeeze=False)
    axes = list(axes[0])
    if has_added:
        for _, xy, title in panels:
            scatter_origin(axes.pop(0), xy, added, f"{title} · kept / added")
    for _, xy, title in panels:
        scatter_types(axes.pop(0), xy, d["labels"], f"{title} · cell type")
    head = f"{s.sample} · {d['n_cells']:,} cells"
    if d["n_added"]:
        head += f", {d['n_added']:,} added by the sync · added-neighbour enrichment {enrichment:.2f}"
    if d["zero_rows"] or d["nan_rows"]:
        head += f" · WARNING: {d['zero_rows']} sampled rows with an all-zero token, {d['nan_rows']} with NaN"
    fig.suptitle(head, fontsize=10, x=0.01, ha="left")
    fig.tight_layout()
    plt.show()

### Summary

One row per mapped slide, highest added-neighbour enrichment first. All-zero or NaN rows in the
sample would mean rows that were never written or are broken. Both should be 0 everywhere.

In [ ]:
summary_df = pd.DataFrame(summary).sort_values("added-neighbour enrichment", ascending=False, na_position="last")
display(summary_df.style.format({"cells": "{:,}", "added by sync": "{:,}", "sampled kept": "{:,}",
                                 "sampled added": "{:,}", "PC1": "{:.1%}", "PC2": "{:.1%}",
                                 "added-neighbour enrichment": "{:.2f}", "seconds": "{:.0f}"})
        .hide(axis="index"))
bad = summary_df[(summary_df["sampled rows with an all-zero token"] > 0) | (summary_df["sampled rows with NaN"] > 0)]
print("No all-zero or NaN rows in any sample." if bad.empty else f"{len(bad)} slide(s) with all-zero / NaN rows: {list(bad['sample'])}")